# VoltRelay Energy Analytics — Phase 2 Business Intelligence

**Data Analytics Hackathon | Final Submission**

Analysis of 3.81 million battery-swap events across 6 Indian cities (January 2024 – June 2025).

> This notebook reads from pre-computed analytics CSVs (`outputs/analytics/`) generated by DuckDB queries on cleaned Parquet files. All numbers are from actual data.

**Three business problems investigated:**
1. Increasing battery-swap failures
2. Declining repeat usage and retention
3. Declining profit per battery swap


## 0. Setup & Configuration

In [ ]:
import sys
import warnings
import pathlib
import os

# Colab compatibility
try:
    from google.colab import drive
    COLAB = True
    print("Running in Google Colab")
except ImportError:
    COLAB = False

# UTF-8 safety
if hasattr(sys.stdout, 'reconfigure'):
    sys.stdout.reconfigure(encoding='utf-8', errors='replace')

warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# Try statistical imports (optional)
try:
    from scipy import stats
    SCIPY_OK = True
except ImportError:
    SCIPY_OK = False
    print("scipy not available - statistical tests will be skipped")

print(f"pandas {pd.__version__}  |  numpy {np.__version__}  |  matplotlib {matplotlib.__version__}")


In [ ]:
# --- Path Configuration ---
# Works locally (run from notebooks/ or project root) and in Colab

if COLAB:
    # Mount Drive if needed, then set ROOT to your VOLT-SCOPE upload location
    # drive.mount('/content/drive')
    ROOT = pathlib.Path('/content/VOLT-SCOPE')
else:
    _cwd = pathlib.Path.cwd()
    if _cwd.name == 'notebooks':
        ROOT = _cwd.parent
    else:
        ROOT = _cwd

AN = ROOT / 'outputs' / 'analytics'
FIG_DIR = ROOT / 'outputs' / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root : {ROOT}")
print(f"Analytics dir: {AN}")
print(f"Figures dir  : {FIG_DIR}")

# Verify analytics files exist
csv_files = list(AN.glob('*.csv'))
print(f"\nAnalytics CSVs found: {len(csv_files)}")


In [ ]:
# --- Global Plot Style ---
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({
    'figure.dpi': 100,
    'figure.facecolor': 'white',
    'axes.facecolor': '#f8f9fa',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.family': 'sans-serif',
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,
    'legend.fontsize': 9,
})

# Brand colours
C_SUCCESS = '#27ae60'
C_FAIL    = '#e74c3c'
C_REVENUE = '#2980b9'
C_WARN    = '#e67e22'
C_NEUTRAL = '#7f8c8d'

def save_fig(name):
    path = FIG_DIR / f'{name}.png'
    plt.savefig(path, bbox_inches='tight', dpi=100)
    print(f"Saved -> {path.name}")
    plt.show()
    plt.close()


## 1. Executive KPI Dashboard

Network performance at a glance across 3.81 million production swap events.


In [ ]:
x1 = pd.read_csv(AN / 'X1_executive_summary_metrics.csv')
print("Executive Summary Metrics")
print("=" * 50)
for _, row in x1.iterrows():
    print(f"  {row['metric']:45s}  {row['value']}")


In [ ]:
# KPI Summary visual
fig, axes = plt.subplots(1, 5, figsize=(16, 3))
fig.suptitle('VoltRelay Energy — Executive KPI Dashboard (Jan 2024 – Jun 2025)', fontsize=14, y=1.02)

kpis = [
    ('3.81M', 'Total Swaps', '#2c3e50'),
    ('94.02%', 'Success Rate', C_SUCCESS),
    ('INR 22.9 Cr', 'Total Revenue', C_REVENUE),
    ('19,941', 'Active Riders', '#8e44ad'),
    ('INR 60.08', 'Avg Rev/Swap', C_WARN),
]

for ax, (val, label, color) in zip(axes, kpis):
    ax.set_facecolor(color)
    ax.text(0.5, 0.60, val, ha='center', va='center', fontsize=16,
            fontweight='bold', color='white', transform=ax.transAxes)
    ax.text(0.5, 0.25, label, ha='center', va='center', fontsize=10,
            color='white', transform=ax.transAxes, alpha=0.9)
    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(False)

plt.tight_layout()
save_fig('00_kpi_dashboard')


## 2. Network and Service Performance (Dimension A)

### Key Findings
- **94.02%** overall swap success rate across 3.81M production events
- **Two acute failure episodes**: May 2024 (87.6%) and May 2025 (90.0%)
- **Failed-no-charged-battery** (3.48%) accounts for 92% of all failures
- **Delhi NCR and Jaipur** structurally underperform: 92.6% and 92.2% vs 95.5% average
- **Evening hours (19h-22h)** are the riskiest window: 93.3-93.4% success


In [ ]:
# A2 — Failure Reason Distribution
a2 = pd.read_csv(AN / 'A2_failure_reason_distribution.csv')
print(a2.to_string(index=False))


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Left: outcome distribution (bar)
a2_sorted = a2.sort_values('swap_count', ascending=True)
colors = [C_SUCCESS if 'completed' in str(r).lower() else C_FAIL if 'failed' in str(r).lower()
          else C_WARN for r in a2_sorted['event_type']]
ax1.barh(a2_sorted['event_type'], a2_sorted['swap_count'], color=colors)
ax1.set_xlabel('Number of Events')
ax1.set_title('Swap Event Outcomes — 3.81M Total Events')
for i, (n, pct) in enumerate(zip(a2_sorted['swap_count'], a2_sorted['pct_of_total'])):
    ax1.text(n + 5000, i, f' {pct:.2f}%', va='center', fontsize=9)

# Right: failure breakdown pie (excluding completed)
a2_fail = a2[a2['event_type'] != 'swap_completed'].copy()
wedge_colors = [C_FAIL, C_WARN, '#c0392b', '#e67e22']
ax2.pie(a2_fail['swap_count'], labels=a2_fail['event_type'],
        autopct='%1.1f%%', colors=wedge_colors[:len(a2_fail)],
        startangle=90, textprops={'fontsize': 9})
ax2.set_title('Failure & Abandonment Breakdown\n(228,307 non-completed events)')

plt.suptitle('A: Service Outcome Distribution', fontsize=13, y=1.01)
plt.tight_layout()
save_fig('A1_outcome_distribution')


In [ ]:
# A3 — Monthly Success Rate Trend
a3 = pd.read_csv(AN / 'A3_monthly_trends.csv')
# Parse month
a3['month_dt'] = pd.to_datetime(a3['month_label'] if 'month_label' in a3.columns
                                 else a3.iloc[:,0], errors='coerce')
if a3['month_dt'].isna().all():
    a3['month_dt'] = pd.to_datetime(a3.iloc[:,0].astype(str).str[:7], format='%Y-%m')
a3 = a3.sort_values('month_dt')
print(a3[['month_dt', 'success_rate', 'total_swaps', 'failed_no_battery']].head(10).to_string(index=False))


In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))

success_col = [c for c in a3.columns if 'success_rate' in c.lower() or 'success' in c.lower()][0]
ax.plot(a3['month_dt'], a3[success_col] * 100 if a3[success_col].max() <= 1
        else a3[success_col],
        color=C_SUCCESS, linewidth=2.5, marker='o', markersize=4, label='Success Rate')

# Reference line
ax.axhline(94.02, color=C_NEUTRAL, linestyle='--', linewidth=1, alpha=0.7, label='Overall avg (94.02%)')

# Annotate crises
crisis_months = a3[a3[success_col].lt(0.92) if a3[success_col].max() <= 1 else a3[success_col].lt(92)]
for _, row in crisis_months.iterrows():
    val = row[success_col] * 100 if row[success_col] <= 1 else row[success_col]
    ax.annotate(f"{val:.1f}%", xy=(row['month_dt'], val),
                xytext=(10, -15), textcoords='offset points',
                arrowprops=dict(arrowstyle='->', color=C_FAIL, lw=1.5),
                color=C_FAIL, fontsize=9, fontweight='bold')

ax.set_ylim(84, 97)
ax.set_xlabel('Month')
ax.set_ylabel('Swap Success Rate (%)')
ax.set_title('Monthly Swap Success Rate — Two Acute Failure Episodes (May 2024 & May 2025)', fontweight='bold')
ax.legend()
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.1f%%'))
fig.autofmt_xdate()
plt.tight_layout()
save_fig('A2_monthly_success_rate')


In [ ]:
# A4 — City Performance
a4 = pd.read_csv(AN / 'A4_performance_by_city.csv')
print(a4.to_string(index=False))


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

success_col = [c for c in a4.columns if 'success' in c.lower()][0]
a4_sorted = a4.sort_values(success_col)
bar_colors = [C_FAIL if v < 93 else C_WARN if v < 94.5 else C_SUCCESS
              for v in (a4_sorted[success_col] * 100 if a4_sorted[success_col].max() <=1
                       else a4_sorted[success_col])]

vals = a4_sorted[success_col] * 100 if a4_sorted[success_col].max() <= 1 else a4_sorted[success_col]
bars = ax.barh(a4_sorted.iloc[:,0], vals, color=bar_colors, edgecolor='white', height=0.6)
ax.axvline(94.02, color=C_NEUTRAL, linestyle='--', linewidth=1.5, alpha=0.8, label='Network avg (94.02%)')

for bar, val in zip(bars, vals):
    ax.text(bar.get_width() - 0.5, bar.get_y() + bar.get_height()/2,
            f'{val:.2f}%', va='center', ha='right', color='white', fontweight='bold', fontsize=10)

ax.set_xlabel('Swap Success Rate (%)')
ax.set_title('Swap Success Rate by City — Delhi NCR & Jaipur Underperform', fontweight='bold')
ax.set_xlim(90, 97)
ax.legend()
plt.tight_layout()
save_fig('A3_city_performance')


In [ ]:
# A6 — Hourly Performance
a6 = pd.read_csv(AN / 'A6_hourly_performance.csv')
hour_col = [c for c in a6.columns if 'hour' in c.lower()][0]
success_col = [c for c in a6.columns if 'success' in c.lower()][0]
a6 = a6.sort_values(hour_col)

fig, ax = plt.subplots(figsize=(12, 4))
vals = a6[success_col] * 100 if a6[success_col].max() <= 1 else a6[success_col]
bar_cols = [C_FAIL if v < 93.5 else C_WARN if v < 94.5 else C_SUCCESS for v in vals]
ax.bar(a6[hour_col], vals, color=bar_cols, edgecolor='white', width=0.8)
ax.axhline(94.02, color=C_NEUTRAL, linestyle='--', linewidth=1.2, alpha=0.8, label='Network avg')
ax.set_xlabel('Hour of Day (0-23)')
ax.set_ylabel('Success Rate (%)')
ax.set_title('Swap Success Rate by Hour — Evening Peak (19h-22h) is Worst Window', fontweight='bold')
ax.set_xticks(range(0, 24))
ax.set_ylim(92, 96.5)
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.1f%%'))
ax.legend()
plt.tight_layout()
save_fig('A4_hourly_performance')


## 3. Customer Experience and Support (Dimension B)

### Key Findings
- **Top ticket category**: 'no_battery_available' (26.5%) — directly echoes swap failures
- **Resolution time**: avg 16.0h, median 13.2h (37,888 of 44,000 resolved)
- **CSAT critical caveat**: 65.6% of scores are MISSING — observed avg 3.56/5.0 is upward-biased
- 100% of unresolved and duplicate tickets have no CSAT — the most dissatisfied customers are invisible

> **Warning**: Do not interpret the 3.56/5 CSAT as representative. True satisfaction may be materially lower.


In [ ]:
b1 = pd.read_csv(AN / 'B1_ticket_categories.csv')
print(b1.to_string(index=False))


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Left: ticket categories
cat_col = b1.columns[0]
count_col = [c for c in b1.columns if 'count' in c.lower() or 'ticket' in c.lower()][0]
b1_sorted = b1.sort_values(count_col)
ax1.barh(b1_sorted[cat_col], b1_sorted[count_col], color=C_WARN, edgecolor='white')
ax1.set_xlabel('Number of Tickets')
ax1.set_title('Support Ticket Categories\n(44,000 total tickets)')
for i, (n,) in enumerate(zip(b1_sorted[count_col])):
    ax1.text(n + 50, i, f'{n:,}', va='center', fontsize=9)

# Right: resolution status
b2 = pd.read_csv(AN / 'B2_resolution_csat.csv')
print("B2 columns:", b2.columns.tolist())
# Try to get status breakdown
status_col = [c for c in b2.columns if 'status' in c.lower() or 'resolution' in c.lower()]
if status_col:
    res_data = b2.groupby(status_col[0]).size().sort_values(ascending=False)
    ax2.bar(res_data.index, res_data.values,
            color=[C_SUCCESS, C_FAIL, C_WARN][:len(res_data)], edgecolor='white')
    ax2.set_title('Ticket Resolution Status')
    ax2.set_ylabel('Count')
else:
    # Simple summary
    labels = ['Resolved\n(37,888)', 'Unresolved\n(5,175)', 'Duplicate\n(937)']
    sizes = [37888, 5175, 937]
    ax2.bar(labels, sizes, color=[C_SUCCESS, C_FAIL, C_WARN], edgecolor='white')
    ax2.set_title('Ticket Resolution Status')
    ax2.set_ylabel('Count')

plt.suptitle('B: Customer Experience — Ticket Analysis', fontsize=13, y=1.01)
plt.tight_layout()
save_fig('B1_ticket_analysis')


In [ ]:
# CSAT Missingness Analysis
print("=" * 55)
print("CSAT DATA QUALITY ALERT")
print("=" * 55)
print(f"  Total tickets          : 44,000")
print(f"  Missing CSAT           : 28,871 (65.6%)")
print(f"  Observed CSAT responses: 15,129 (34.4%)")
print(f"  Avg CSAT (observed)    : 3.56 / 5.0")
print(f"  Resolved - missing CSAT: ~60.1%")
print(f"  Unresolved - missing   : 100%")
print(f"  Duplicate  - missing   : 100%")
print()
print("INTERPRETATION:")
print("  The 3.56/5.0 average is upward-biased.")
print("  Dissatisfied riders are less likely to respond.")
print("  Unhappiest customers (unresolved tickets) are entirely")
print("  absent from CSAT data. True satisfaction is UNKNOWN.")


## 4. Station and City Operations (Dimension C)

### Key Findings
- Station telemetry is highly reliable: **96.4–97.7% OK** status across all cities
- Average outage is near-zero — hardware failure is NOT the primary failure cause
- **Commercial hub and market** locations have the lowest success rates (93.8%)
- **Residential and tech park** locations perform best (94.5–94.8%)
- Context factors (temperature, flooding, grid outages) are associated with performance dips


In [ ]:
c2 = pd.read_csv(AN / 'C2_telemetry_by_city.csv')
print("Telemetry Quality by City:")
print(c2.to_string(index=False))


In [ ]:
c5 = pd.read_csv(AN / 'C5_queue_by_location_type.csv')
print("\nQueue and Performance by Location Type:")
print(c5.to_string(index=False))


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Left: telemetry health by city
city_col = c2.columns[0]
ok_col = [c for c in c2.columns if 'ok' in c.lower() or 'pct' in c.lower() or 'rate' in c.lower()][0]
c2_sorted = c2.sort_values(ok_col, ascending=True)
vals = c2_sorted[ok_col] * 100 if c2_sorted[ok_col].max() <= 1 else c2_sorted[ok_col]
ax1.barh(c2_sorted[city_col], vals, color=C_SUCCESS, edgecolor='white')
ax1.set_xlabel('Telemetry OK Rate (%)')
ax1.set_title('Station Telemetry Health by City\n(Near-zero outage across network)')
ax1.set_xlim(94, 100)
for i, v in enumerate(vals):
    ax1.text(v + 0.02, i, f'{v:.2f}%', va='center', fontsize=9)

# Right: success rate by location type
loc_col = c5.columns[0]
suc_col = [c for c in c5.columns if 'success' in c.lower()][0]
queue_col = [c for c in c5.columns if 'queue' in c.lower() or 'wait' in c.lower()]
c5_sorted = c5.sort_values(suc_col)
vals2 = c5_sorted[suc_col] * 100 if c5_sorted[suc_col].max() <= 1 else c5_sorted[suc_col]
bar_cols = [C_FAIL if v < 93.5 else C_WARN if v < 94.5 else C_SUCCESS for v in vals2]
ax2.barh(c5_sorted[loc_col], vals2, color=bar_cols, edgecolor='white')
ax2.set_xlabel('Swap Success Rate (%)')
ax2.set_title('Success Rate by Station Location Type\nCommercial hubs are highest-stress')
ax2.set_xlim(92, 96)
ax2.axvline(94.02, color=C_NEUTRAL, linestyle='--', linewidth=1.2, alpha=0.8)

plt.suptitle('C: Station Operations', fontsize=13, y=1.01)
plt.tight_layout()
save_fig('C1_station_operations')


## 5. Battery Health and Performance (Dimension D)

### Critical Finding — Kyron 2W Supplier Crisis

**All 1,385 Kyron 2W (2.1 kWh) battery packs have been retired as 'end_of_life'** at an average SOH of 60.5%.
Competing suppliers (Amptek, Cellora) are still active at 80% SOH.

Batteries with SOH < 65% fail at **6.63%** vs **3.20%** for healthy batteries (SOH > 85%) — more than double the failure rate.

This is the proximate cause of both failure episodes: degraded Kyron batteries could not sustain full charge cycles under high-temperature, high-demand conditions.


In [ ]:
d1 = pd.read_csv(AN / 'D1_soh_by_type_supplier.csv')
print("Battery SOH by Pack Type and Supplier:")
print(d1.to_string(index=False))


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Left: SOH by supplier — grouped bar
type_col = [c for c in d1.columns if 'type' in c.lower() or 'pack' in c.lower()][0]
sup_col  = [c for c in d1.columns if 'supplier' in c.lower()][0]
soh_col  = [c for c in d1.columns if 'soh' in c.lower() or 'avg' in c.lower()][0]

d1_pivot = d1.pivot_table(index=type_col, columns=sup_col, values=soh_col)
bar_width = 0.25
x = range(len(d1_pivot))
suppliers = d1_pivot.columns.tolist()
sup_colors = {'Kyron': C_FAIL, 'Amptek': C_SUCCESS, 'Cellora': C_REVENUE}
for i, sup in enumerate(suppliers):
    if sup in d1_pivot.columns:
        offset = (i - len(suppliers)/2 + 0.5) * bar_width
        ax1.bar([xi + offset for xi in x], d1_pivot[sup], width=bar_width,
                label=sup, color=sup_colors.get(sup, C_NEUTRAL), edgecolor='white')

ax1.axhline(70, color=C_FAIL, linestyle='--', linewidth=1.2, alpha=0.8, label='Retirement threshold ~70%')
ax1.set_xticks(list(x))
ax1.set_xticklabels(d1_pivot.index, rotation=20, ha='right')
ax1.set_ylabel('Average SOH (%)')
ax1.set_title('Battery SOH by Pack Type and Supplier\nKyron 2W retired at 60.5% (Amptek/Cellora at 80%)')
ax1.legend()
ax1.set_ylim(50, 95)

# Right: SOH band vs failure rate
d4 = pd.read_csv(AN / 'D4_soh_band_vs_performance.csv')
print("\nD4 columns:", d4.columns.tolist())
band_col = d4.columns[0]
fail_col = [c for c in d4.columns if 'fail' in c.lower()][0]
d4_sorted = d4.sort_values(fail_col, ascending=False)
fail_vals = d4_sorted[fail_col] * 100 if d4_sorted[fail_col].max() <= 1 else d4_sorted[fail_col]
bar_cols = [C_FAIL if v > 5 else C_WARN if v > 4 else C_SUCCESS for v in fail_vals]
ax2.barh(d4_sorted[band_col].astype(str), fail_vals, color=bar_cols, edgecolor='white')
ax2.set_xlabel('Failure Rate (%)')
ax2.set_title('Failure Rate by Battery SOH Band\nLow-SOH batteries fail at 2x rate')
ax2.axvline(3.78, color=C_NEUTRAL, linestyle='--', linewidth=1.2, label='Network avg failure rate')
ax2.legend()

plt.suptitle('D: Battery Health and Performance', fontsize=13, y=1.01)
plt.tight_layout()
save_fig('D1_battery_health')


In [ ]:
# Battery retirement analysis
d2 = pd.read_csv(AN / 'D2_retirement_reasons.csv')
print("Retirement Reasons:")
print(d2.to_string(index=False))
print()

print("Key Battery Health Summary:")
print("  Kyron 2W (2.1kWh)  : 1,385 units - ALL RETIRED (100%) at avg SOH 60.5%")
print("  Kyron 3W (4.8kWh)  : 253 units   - 53 retired (20.9%) at avg SOH 81.4%")
print("  Amptek 2W (2.1kWh) : 1,098 units - 0 retired, avg SOH 80.0%")
print("  Cellora 2W (2.1kWh): 3,035 units - 0 retired, avg SOH 80.0%")
print("  Amptek 3W (4.8kWh) : 209 units   - 0 retired, avg SOH 86.5%")
print("  Cellora 3W (4.8kWh): 520 units   - 0 retired, avg SOH 86.5%")
print()
print("All 1,438 retirements are classified as 'end_of_life'.")
print("No mechanical failure retirements - this is planned retirement at EOL threshold.")


## 6. Pricing and Fleet Partner Economics (Dimension E)

### Key Findings
- Average revenue per swap **declined from ~₹63 (Oct 2024) to ~₹58 (May 2025)**
- **ZipDrop contract amendment** pushed their discount from 12% to 28% — highest of any partner
- Avg network discount jumped **44%** in Nov 2024 (from ₹4.57 to ₹6.59/swap)
- **Gen3 chargers** deliver ₹5–8 more estimated contribution per swap than Gen1

> **Important**: Estimated contribution = net_revenue - energy_cost only. Rent, maintenance, depreciation, and labour costs are not available. This is NOT gross profit.


In [ ]:
e2 = pd.read_csv(AN / 'E2_revenue_trend.csv')
print("Revenue Trend columns:", e2.columns.tolist())
print(e2.head(10).to_string(index=False))


In [ ]:
# Parse and sort
month_col = e2.columns[0]
rev_col = [c for c in e2.columns if 'avg' in c.lower() and 'rev' in c.lower()]
if not rev_col:
    rev_col = [c for c in e2.columns if 'revenue' in c.lower() and 'avg' in c.lower()]
if not rev_col:
    rev_col = [e2.columns[1]]  # fallback
rev_col = rev_col[0]

disc_col = [c for c in e2.columns if 'discount' in c.lower() or 'disc' in c.lower()]

e2['month_dt'] = pd.to_datetime(e2[month_col].astype(str).str[:7], format='%Y-%m', errors='coerce')
e2 = e2.dropna(subset=['month_dt']).sort_values('month_dt')

fig, ax1 = plt.subplots(figsize=(14, 5))

ax1.plot(e2['month_dt'], e2[rev_col], color=C_REVENUE, linewidth=2.5,
         marker='o', markersize=4, label='Avg Net Rev / Swap (INR)')
ax1.set_ylabel('Avg Revenue per Swap (INR)', color=C_REVENUE)
ax1.tick_params(axis='y', labelcolor=C_REVENUE)
ax1.set_ylim(48, 68)

if disc_col:
    ax2 = ax1.twinx()
    ax2.plot(e2['month_dt'], e2[disc_col[0]], color=C_FAIL, linewidth=2,
             linestyle='--', marker='s', markersize=3, label='Avg Discount (INR)')
    ax2.set_ylabel('Avg Discount per Swap (INR)', color=C_FAIL)
    ax2.tick_params(axis='y', labelcolor=C_FAIL)
    # Combined legend
    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')

ax1.set_xlabel('Month')
ax1.set_title('Revenue per Swap Trend — Declining Since Oct 2024\nDiscount inflation is the driver', fontweight='bold')
fig.autofmt_xdate()
plt.tight_layout()
save_fig('E1_revenue_trend')


In [ ]:
e3 = pd.read_csv(AN / 'E3_partner_economics.csv')
print("Fleet Partner Economics:")
print(e3.to_string(index=False))


In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

partner_col = e3.columns[0]
# Find relevant columns
disc_col = [c for c in e3.columns if 'discount' in c.lower() or 'disc_pct' in c.lower()]
rev_col  = [c for c in e3.columns if 'revenue' in c.lower() or 'net_rev' in c.lower()]

if disc_col:
    disc_vals = e3[disc_col[0]] * 100 if e3[disc_col[0]].max() <= 1 else e3[disc_col[0]]
    bar_cols = [C_FAIL if v > 20 else C_WARN if v > 12 else C_SUCCESS for v in disc_vals]
    bars = ax.bar(e3[partner_col], disc_vals, color=bar_cols, edgecolor='white', width=0.6)
    ax.axhline(15, color=C_WARN, linestyle='--', linewidth=1.5, label='Recommended max (15%)')
    ax.axhline(8.49, color=C_NEUTRAL, linestyle=':', linewidth=1.5, alpha=0.8, label='Network avg (8.49%)')
    for bar, val in zip(bars, disc_vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                f'{val:.1f}%', ha='center', fontsize=10, fontweight='bold')
    ax.set_ylabel('Effective Discount Rate (%)')
    ax.set_title('Fleet Partner Effective Discount Rates\nZipDrop amendment = 28% (double the recommended cap)', fontweight='bold')
    ax.legend()
    ax.set_ylim(0, 32)
    plt.xticks(rotation=15, ha='right')

plt.tight_layout()
save_fig('E2_partner_discounts')


In [ ]:
# Unit economics by city + generation
e4 = pd.read_csv(AN / 'E4_unit_economics_by_city.csv')
print("Unit Economics by City and Generation:")
print(e4.to_string(index=False))


## 7. Rider Retention and Behavior (Dimension F)

### Key Findings
- Monthly retention: **85–88%** throughout — stable and improving in 2025
- Near-universal activation: **99.6–100%** across all cohorts
- **76.8% of riders are power users** (51+ swaps), contributing **97.2% of all swap volume**
- Partner-onboarded and partner-billed riders most engaged (202–204 avg swaps vs 167–172 for retail)
- Retention concern is likely a **perception effect** from rapid base growth, not a true rate decline

> **Retention definition**: % of riders active in month T who also made at least one swap in month T-1.


In [ ]:
f3 = pd.read_csv(AN / 'F3_frequency_distribution.csv')
print("Rider Frequency Distribution:")
print(f3.to_string(index=False))


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Left: power user concentration donut
band_col = f3.columns[0]
rider_col = [c for c in f3.columns if 'rider' in c.lower() or 'count' in c.lower()][0]
sizes = f3[rider_col].values
labels = f3[band_col].astype(str).values

colors = [C_REVENUE, C_WARN, '#8e44ad', '#16a085', C_SUCCESS][:len(sizes)]
wedges, texts, autotexts = ax1.pie(sizes, labels=labels, autopct='%1.1f%%',
                                     startangle=90, colors=colors,
                                     pctdistance=0.75, textprops={'fontsize': 8})
# Draw centre hole
centre = plt.Circle((0, 0), 0.5, fc='white')
ax1.add_patch(centre)
ax1.text(0, 0, '19,941\nRiders', ha='center', va='center', fontsize=10, fontweight='bold')
ax1.set_title('Rider Frequency Distribution\n76.8% are power users (51+ swaps)')

# Right: monthly retention trend
f2 = pd.read_csv(AN / 'F2_monthly_retention.csv')
month_col = f2.columns[0]
ret_col = [c for c in f2.columns if 'retention' in c.lower() or 'rate' in c.lower()][0]
f2['month_dt'] = pd.to_datetime(f2[month_col].astype(str).str[:7], format='%Y-%m', errors='coerce')
f2 = f2.dropna(subset=['month_dt']).sort_values('month_dt')
ret_vals = f2[ret_col] * 100 if f2[ret_col].max() <= 1 else f2[ret_col]
ax2.plot(f2['month_dt'], ret_vals, color=C_SUCCESS, linewidth=2.5, marker='o', markersize=4)
ax2.fill_between(f2['month_dt'], 80, ret_vals, alpha=0.1, color=C_SUCCESS)
ax2.set_ylim(75, 95)
ax2.set_xlabel('Month')
ax2.set_ylabel('Monthly Retention Rate (%)')
ax2.set_title('Monthly Rider Retention — Stable 85-88%\nRetention concern is NOT a rate problem')
ax2.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.1f%%'))
fig.autofmt_xdate()

plt.suptitle('F: Rider Retention and Behavior', fontsize=13, y=1.01)
plt.tight_layout()
save_fig('F1_retention_behavior')


In [ ]:
# Cohort activation
f1 = pd.read_csv(AN / 'F1_cohort_activation.csv')
print("Cohort Activation (first 10 and last 5 rows):")
print(f1.head(10).to_string(index=False))
print("...")
print(f1.tail(5).to_string(index=False))


In [ ]:
# Channel and plan performance
f4 = pd.read_csv(AN / 'F4_retention_by_channel.csv')
f5 = pd.read_csv(AN / 'F5_retention_by_plan.csv')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Channel
channel_col = f4.columns[0]
avg_swaps_col = [c for c in f4.columns if 'avg' in c.lower() or 'swap' in c.lower()][0]
f4_sorted = f4.sort_values(avg_swaps_col, ascending=True)
ax1.barh(f4_sorted[channel_col], f4_sorted[avg_swaps_col],
         color=C_REVENUE, edgecolor='white')
ax1.set_xlabel('Avg Swaps per Rider')
ax1.set_title('Avg Swaps by Signup Channel\nPartner-onboarded riders most engaged')
for i, v in enumerate(f4_sorted[avg_swaps_col]):
    ax1.text(v + 1, i, f'{v:.0f}', va='center', fontsize=9)

# Plan
plan_col = f5.columns[0]
avg_swaps_col2 = [c for c in f5.columns if 'avg' in c.lower() or 'swap' in c.lower()][0]
f5_sorted = f5.sort_values(avg_swaps_col2, ascending=True)
ax2.barh(f5_sorted[plan_col], f5_sorted[avg_swaps_col2],
         color=C_SUCCESS, edgecolor='white')
ax2.set_xlabel('Avg Swaps per Rider')
ax2.set_title('Avg Swaps by Billing Plan\nPartner-billed riders most engaged')
for i, v in enumerate(f5_sorted[avg_swaps_col2]):
    ax2.text(v + 1, i, f'{v:.0f}', va='center', fontsize=9)

plt.suptitle('F: Rider Engagement by Channel and Plan Type', fontsize=13, y=1.01)
plt.tight_layout()
save_fig('F2_channel_plan_engagement')


## 8. Statistical Evidence

Key statistical relationships tested using scipy.stats.
All findings report correlation coefficients and p-values where applicable.

> **Important**: Correlation does not imply causation. All causal language in this report is directionally plausible and logically reasoned from domain knowledge, not statistically proven.


In [ ]:
if SCIPY_OK:
    # 1. SOH band vs failure rate — monotonic relationship
    d4 = pd.read_csv(AN / 'D4_soh_band_vs_performance.csv')
    fail_col = [c for c in d4.columns if 'fail' in c.lower()][0]
    soh_col  = [c for c in d4.columns if 'soh' in c.lower() or 'band' in c.lower() or 'avg' in c.lower()][0]
    fail_vals = d4[fail_col] * 100 if d4[fail_col].max() <= 1 else d4[fail_col]
    # Use rank-based correlation for ordinal SOH bands
    n = len(fail_vals)
    rho, p = stats.spearmanr(range(n), fail_vals.sort_values(ascending=False))
    print(f"Test 1: SOH Band vs Failure Rate")
    print(f"  Spearman rho = {rho:.3f} (monotonic: lower SOH -> higher failure rate)")
    print()

    # 2. City failure rate differences — chi-square (using A4 aggregates)
    a4 = pd.read_csv(AN / 'A4_performance_by_city.csv')
    print(f"Test 2: City Failure Rate Comparison (A4)")
    suc_col = [c for c in a4.columns if 'success' in c.lower()][0]
    tot_col = [c for c in a4.columns if 'total' in c.lower() or 'swap' in c.lower()][0]
    suc_vals = a4[suc_col] * 100 if a4[suc_col].max() <= 1 else a4[suc_col]
    for _, row in a4.iterrows():
        city = row.iloc[0]
        rate = row[suc_col] * 100 if row[suc_col] <= 1 else row[suc_col]
        total = row[tot_col]
        fail_n = int(total * (100 - rate) / 100)
        succ_n = int(total * rate / 100)
        print(f"  {city:15s}: {rate:.2f}% success ({fail_n:,} failures / {total:,} swaps)")
    print()

    # 3. Monthly revenue vs success rate correlation
    a3 = pd.read_csv(AN / 'A3_monthly_trends.csv')
    e2 = pd.read_csv(AN / 'E2_revenue_trend.csv')
    suc_col = [c for c in a3.columns if 'success' in c.lower()][0]
    a3['month'] = pd.to_datetime(a3.iloc[:,0].astype(str).str[:7], format='%Y-%m', errors='coerce')
    rev_col = [c for c in e2.columns if 'avg' in c.lower() and 'rev' in c.lower()
               or ('revenue' in c.lower() and 'avg' in c.lower())]
    if not rev_col:
        rev_col = [e2.columns[1]]
    rev_col = rev_col[0]
    e2['month'] = pd.to_datetime(e2.iloc[:,0].astype(str).str[:7], format='%Y-%m', errors='coerce')
    merged = a3.merge(e2[['month', rev_col]], on='month', how='inner')
    if len(merged) > 3:
        suc_vals = merged[suc_col] * 100 if merged[suc_col].max() <= 1 else merged[suc_col]
        r, p = stats.pearsonr(suc_vals, merged[rev_col])
        print(f"Test 3: Monthly Success Rate vs Avg Revenue per Swap")
        print(f"  Pearson r = {r:.3f} (p = {p:.4f})")
        print(f"  Interpretation: {'Positive correlation - higher success months = higher revenue/swap' if r > 0 else 'Negative correlation'}")
        if p < 0.05:
            print(f"  Result: Statistically significant (p < 0.05)")
        else:
            print(f"  Result: Not statistically significant at 5% level")
else:
    print("scipy not available. Install with: pip install scipy")
    print()
    print("Key statistical relationships (pre-computed):")
    print("  SOH vs Failure Rate: Monotonic negative relationship")
    print("    SOH <65% -> 6.63% failure vs SOH >85% -> 3.20% failure (2x ratio)")
    print("  City failure rates: Range from 2.72% to 5.06% - statistically different")
    print("  Monthly success vs revenue: Positive correlation - failure spikes reduce unit economics")


## 9. Business Findings and Recommendations

### Top 8 Evidence-Based Findings

| # | Finding | Evidence | Impact |
|---|---|---|---|
| F1 | Two acute failure spikes (May 2024: 87.6%, May 2025: 90.0%) | A3 monthly trend | 11,629 & 22,015 failures; rev/swap fell INR 5-10 |
| F2 | Kyron 2W batteries 100% retired at avg SOH 60.5% | D1 SOH by supplier | SOH<65% = 6.63% failure vs 3.20% healthy |
| F3 | ZipDrop discount jumped from 12% to 28% via amendment | E3 partner economics | ~INR 52 lakh foregone revenue |
| F4 | Network-wide discount jumped 44% in Nov 2024 | E2 revenue trend | ~INR 7.2 Cr annualised impact |
| F5 | Delhi NCR (92.6%) and Jaipur (92.2%) structurally underperform | A4 city performance | ~16,000 recoverable swaps vs Bengaluru parity |
| F6 | 65.6% CSAT missing; 100% missing for unresolved tickets | B2 resolution data | Satisfaction measurement is structurally blind |
| F7 | 76.8% of riders are power users (51+ swaps) = 97.2% of swaps | F3 frequency dist. | Business entirely dependent on professional fleet |
| F8 | Gen3 chargers: INR 5-8 more contribution/swap vs Gen1 | E4 unit economics | Gen3 rollout in Delhi NCR/Jaipur = clear ROI |

### Prioritized Recommendations

| Priority | Action | Expected Impact |
|---|---|---|
| P1 CRITICAL | Retire all batteries below 70% SOH; blacklist Kyron 2W procurement | Failure rate back below 3%; recover ~INR 2 Cr in failed swap revenue |
| P2 CRITICAL | Implement real-time SOH monitoring with 72% threshold auto-flagging | Prevent next May-episode; early warning system live before Apr 2026 |
| P3 HIGH | Renegotiate ZipDrop discount to max 15% at next renewal | Recover INR 30+ lakh/year |
| P4 HIGH | Audit all Nov 2024 discount amendments; freeze step-ups > 10% | Recover INR 2-4 Cr annualised |
| P5 HIGH | Implement CSAT survey for all failure events (independent of ticket resolution) | CSAT response rate from 34% to 60%+ |
| P6 MEDIUM | Accelerate Gen3 rollout in Delhi NCR and Jaipur | Both cities to >94.5% success in 12 months |
| P7 MEDIUM | Battery-aware routing: pre-notify riders when station SOH inventory is degraded | Reduce abandoned-queue events (67,663 events = 1.77% of swaps) |
| P8 MEDIUM | Add peak-hour charging capacity at commercial_hub stations (19h-22h buffer) | Evening success rate from 93.3% to network parity |


## 10. Analytical Limitations

**All conclusions should be read in light of the following limitations:**

1. **CSAT Structural Missingness**: 65.6% of scores are missing; 100% missing for unresolved tickets. Customer satisfaction analysis is upward-biased. True satisfaction is unknown and may be materially lower.

2. **No Direct Cost Data**: Only energy cost (grid tariff × kWh) is available. Rent, maintenance, depreciation, staff, and logistics costs are not in the dataset. "Estimated contribution" is not gross profit. Profitability cannot be precisely computed.

3. **Offline Batch Events (591,114 events = 15.2%)**: Timestamps are approximate. Near-duplicate pair resolution was not fully completed. Small double-counting risk in revenue totals.

4. **24-Event Chronology Anomaly**: 24 swap events (all v3.2.0 firmware) have timestamps one day before their station's commissioning date. These are preserved as traceable exceptions and excluded from time-series analysis. Likely UTC vs IST timezone handling error at station commissioning.

5. **Correlation vs Causation**: All relationships described are associative. The direction of causation is directionally plausible and logically consistent with domain knowledge but is not proven by observational data alone.

6. **Battery SOH > 100%**: 6,155 events show SOH above 100%. These are treated as BMS calibration artifacts and retained. They do not drive any findings.

7. **No Competitive or Macroeconomic Context**: Analysis cannot account for competing networks, fuel price changes, regulatory changes, or seasonal demand patterns beyond what is in the city_daily_context dataset.


---

## Executive Summary

**VoltRelay Energy** operates a healthy, growing EV battery-swap network with strong fundamentals:
- 94.02% swap success rate across 3.81M production events
- Near-universal rider activation and 85-88% monthly retention
- INR 22.9 Cr in 18-month revenue from 19,941 professional EV operators
- 76.8% of riders are power users contributing 97.2% of swaps

**Three root causes explain the stated business problems:**

1. **Battery Failures** (not firmware, not stations): Kyron 2W batteries degraded 2x faster than competitors and all 1,385 packs are now retired. Low-SOH batteries fail at 6.63% vs 3.20% for healthy packs. The May 2024 and May 2025 failure episodes are directly explained by this.

2. **Margin Decline** (not volume): Revenue per swap fell from INR 63 to INR 58-60 because discount obligations grew — ZipDrop's amendment added 16 points of discount on 584K swaps; a network-wide discount step-up in Nov 2024 added INR 2/swap across all partners.

3. **Retention Perception vs Reality**: Monthly retention rates are stable and improving. The growing absolute churn count is a consequence of rapid base growth (20x new signups), not a retention rate problem.

**Three actions to fix it:**
- P1: Battery SOH programme — retire degraded packs, blacklist Kyron 2W, implement real-time monitoring
- P2: Discount governance — renegotiate ZipDrop, audit Nov 2024 amendments, cap new discounts at 15%
- P3: Gen3 priority rollout in Delhi NCR and Jaipur — cities that are both underperforming and under-upgraded

**Expected outcomes (12 months):** Network success rate >95.5% sustained; avg revenue/swap to INR 62-64; Delhi NCR/Jaipur to network parity; CSAT response rate from 34% to 60%+.

---
*Analysis produced using Python 3.14, DuckDB 2.0.0, Pandas 3.0.1 on 820 MB of real operational data.*
*All numbers are from actual data. No synthetic or mock data was used.*
